In [1]:
import base64
import importlib.util
import io
import json
import math
import random
import sys
import types
import warnings
from dataclasses import dataclass
from pathlib import Path
from types import MethodType
from typing import Dict, Optional

import av
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display
from peft import LoraConfig, get_peft_model
from safetensors.torch import load_file
from tqdm.auto import trange


DATASET_ROOT = Path("/home/azureuser/datasets/navier_fast")
WAN_REPO_ROOT = Path("/home/azureuser/physics/navier/Wan2.2")
WAN_CHECKPOINT_DIR = Path("/home/azureuser/Wan2.2-TI2V-5B")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PARAM_DTYPE = torch.bfloat16
TRAIN_DTYPE = torch.bfloat16

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 3e-5
LR_WARMUP_STEPS = 500
MAX_STEPS = 40000
VALIDATION_EVERY = 1000
VALIDATION_COUNT = 100
VALIDATION_INFERENCE_STEPS = 25
VALIDATION_ERROR_VMAX = 0.45
LOSS_EMA_BETA = 0.98

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

REQUIRE_RHO = True
WAN_NUM_TRAIN_TIMESTEPS = 1000
WAN_SAMPLE_SHIFT = 5.0
WAN_SAMPLE_SOLVER = "unipc"
RANDOM_SEED = 1234

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


def _load_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load {module_name} from {module_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def load_wan_model_module(wan_repo_root: Path = WAN_REPO_ROOT):
    """Load Wan modules without importing wan/__init__.py optional audio deps."""
    wan_root = Path(wan_repo_root) / "wan"
    modules_root = wan_root / "modules"

    wan_pkg = types.ModuleType("wan")
    wan_pkg.__path__ = [str(wan_root)]
    modules_pkg = types.ModuleType("wan.modules")
    modules_pkg.__path__ = [str(modules_root)]
    sys.modules["wan"] = wan_pkg
    sys.modules["wan.modules"] = modules_pkg

    _load_module("wan.modules.attention", modules_root / "attention.py")
    return _load_module("wan.modules.model", modules_root / "model.py")


def torch_sdpa_attention(
    q,
    k,
    v,
    q_lens=None,
    k_lens=None,
    dropout_p=0.0,
    softmax_scale=None,
    q_scale=None,
    causal=False,
    window_size=(-1, -1),
    deterministic=False,
    dtype=torch.bfloat16,
    version=None,
):
    """Wan-compatible attention using torch SDPA.

    The local flash_attn_interface build has an API mismatch with this Wan2.2
    checkout. Our dataset uses fixed-size latent sequences, so no padding mask is
    needed and SDPA is a reliable in-memory fallback.
    """
    if q_scale is not None:
        q = q * q_scale
    q = q.transpose(1, 2).to(dtype)
    k = k.transpose(1, 2).to(dtype)
    v = v.transpose(1, 2).to(dtype)
    out = F.scaled_dot_product_attention(
        q,
        k,
        v,
        dropout_p=dropout_p,
        is_causal=causal,
        scale=softmax_scale,
    )
    return out.transpose(1, 2).contiguous()


def load_wan_vae_class(wan_repo_root: Path = WAN_REPO_ROOT):
    return _load_module(
        "wan.modules.vae2_2_local_for_train",
        Path(wan_repo_root) / "wan" / "modules" / "vae2_2.py",
    ).Wan2_2_VAE


def load_wan_scheduler_modules(wan_repo_root: Path = WAN_REPO_ROOT):
    """Load Wan flow-matching schedulers without importing wan/__init__.py."""
    wan_root = Path(wan_repo_root) / "wan"
    utils_root = wan_root / "utils"

    wan_pkg = sys.modules.get("wan") or types.ModuleType("wan")
    wan_pkg.__path__ = [str(wan_root)]
    utils_pkg = sys.modules.get("wan.utils") or types.ModuleType("wan.utils")
    utils_pkg.__path__ = [str(utils_root)]
    sys.modules["wan"] = wan_pkg
    sys.modules["wan.utils"] = utils_pkg

    fm_solvers = _load_module("wan.utils.fm_solvers", utils_root / "fm_solvers.py")
    fm_solvers_unipc = _load_module("wan.utils.fm_solvers_unipc", utils_root / "fm_solvers_unipc.py")
    return fm_solvers, fm_solvers_unipc


wan_model_module = load_wan_model_module()
wan_model_module.flash_attention = torch_sdpa_attention
WanModel = wan_model_module.WanModel
sinusoidal_embedding_1d = wan_model_module.sinusoidal_embedding_1d

wan_fm_solvers_module, wan_fm_solvers_unipc_module = load_wan_scheduler_modules()
FlowDPMSolverMultistepScheduler = wan_fm_solvers_module.FlowDPMSolverMultistepScheduler
FlowUniPCMultistepScheduler = wan_fm_solvers_unipc_module.FlowUniPCMultistepScheduler
get_sampling_sigmas = wan_fm_solvers_module.get_sampling_sigmas
retrieve_timesteps = wan_fm_solvers_module.retrieve_timesteps


def load_wan_transformer(
    checkpoint_dir: Path = WAN_CHECKPOINT_DIR,
    device: torch.device = DEVICE,
    dtype: torch.dtype = PARAM_DTYPE,
) -> WanModel:
    """Load Wan2.2 TI2V-5B transformer from sharded safetensors."""
    checkpoint_dir = Path(checkpoint_dir)
    try:
        model = WanModel.from_pretrained(
            str(checkpoint_dir),
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        )
    except Exception as exc:
        warnings.warn(f"Diffusers from_pretrained failed ({exc}); falling back to manual shard load.")
        config = json.loads((checkpoint_dir / "config.json").read_text())
        config.pop("_class_name", None)
        config.pop("_diffusers_version", None)
        model = WanModel(**config)
        state = {}
        for shard in sorted(checkpoint_dir.glob("diffusion_pytorch_model-*.safetensors")):
            state.update(load_file(str(shard)))
        model.load_state_dict(state, strict=True)
        del state

    model.eval().requires_grad_(False)
    return model.to(device=device, dtype=dtype)


def attach_physics_adaln_forward(model: WanModel) -> WanModel:
    """Add log-nu/log-rho conditioning with one full-width AdaLN0 projector per block."""
    dim = model.dim
    model.physics_adaln = nn.ModuleList([
        nn.Sequential(
            nn.Linear(2, 32),
            nn.SiLU(),
            nn.Linear(32, 6 * dim),
        )
        for _ in model.blocks
    ]).to(device=next(model.parameters()).device, dtype=torch.float32)
    for adaln in model.physics_adaln:
        nn.init.zeros_(adaln[-1].weight)
        nn.init.zeros_(adaln[-1].bias)

    def forward_with_physics(self, x, t, context, seq_len, y=None, physics=None):
        if self.model_type == "i2v":
            assert y is not None

        device = self.patch_embedding.weight.device
        if self.freqs.device != device:
            self.freqs = self.freqs.to(device)

        if y is not None:
            x = [torch.cat([u, v], dim=0) for u, v in zip(x, y)]

        x = [self.patch_embedding(u.unsqueeze(0)) for u in x]
        grid_sizes = torch.stack([torch.tensor(u.shape[2:], dtype=torch.long, device=device) for u in x])
        x = [u.flatten(2).transpose(1, 2) for u in x]
        seq_lens = torch.tensor([u.size(1) for u in x], dtype=torch.long, device=device)
        assert seq_lens.max() <= seq_len
        x = torch.cat([
            torch.cat([u, u.new_zeros(1, seq_len - u.size(1), u.size(2))], dim=1)
            for u in x
        ])

        if t.dim() == 1:
            t = t.expand(t.size(0), seq_len)
        with torch.amp.autocast("cuda", dtype=torch.float32):
            bt = t.size(0)
            t_flat = t.flatten()
            e = self.time_embedding(
                sinusoidal_embedding_1d(self.freq_dim, t_flat)
                .unflatten(0, (bt, seq_len))
                .float()
                .to(device)
            )
            e0 = self.time_projection(e).unflatten(2, (6, self.dim))
            physics_e0 = None
            if physics is not None:
                physics = physics.to(device=device, dtype=torch.float32)
                if physics.shape != (bt, 2):
                    raise ValueError(f"physics must have shape ({bt}, 2), got {tuple(physics.shape)}")
                physics_e0 = torch.stack([adaln(physics) for adaln in self.physics_adaln], dim=1)
                physics_e0 = physics_e0.view(bt, len(self.blocks), 1, 6, self.dim)

        context_lens = None
        context = self.text_embedding(
            torch.stack([
                torch.cat([u, u.new_zeros(self.text_len - u.size(0), u.size(1))])
                for u in context
            ])
        )

        kwargs = dict(
            e=e0,
            seq_lens=seq_lens,
            grid_sizes=grid_sizes,
            freqs=self.freqs,
            context=context,
            context_lens=context_lens,
        )
        for block_index, block in enumerate(self.blocks):
            kwargs["e"] = e0 if physics_e0 is None else e0 + physics_e0[:, block_index]
            x = block(x, **kwargs)

        x = self.head(x, e)
        x = self.unpatchify(x, grid_sizes)
        return [u.float() for u in x]

    model.forward = MethodType(forward_with_physics, model)
    return model


def add_lora_to_wan(model: WanModel):
    """PEFT LoRA on transformer block output linears."""
    config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["ffn.2"],
    )
    peft_model = get_peft_model(model, config)
    base = peft_model.get_base_model()
    for p in base.physics_adaln.parameters():
        p.requires_grad_(True)
    return peft_model


@dataclass
class NavierSample:
    folder: Path
    latents_path: Path
    metadata_path: Path
    video_path: Path
    nu: float
    rho: float


def discover_samples(dataset_root: Path = DATASET_ROOT, require_rho: bool = REQUIRE_RHO):
    samples = []
    for latents_path in sorted(Path(dataset_root).glob("*/latents.safetensors")):
        folder = latents_path.parent
        metadata_path = folder / "metadata.json"
        video_path = folder / "video.mp4"
        if not metadata_path.exists() or not video_path.exists():
            continue
        try:
            metadata = json.loads(metadata_path.read_text())
        except json.JSONDecodeError:
            continue
        if require_rho and "rho" not in metadata:
            continue
        samples.append(
            NavierSample(
                folder=folder,
                latents_path=latents_path,
                metadata_path=metadata_path,
                video_path=video_path,
                nu=float(metadata["nu"]),
                rho=float(metadata["rho"] if "rho" in metadata else 1.0),
            )
        )
    if len(samples) <= VALIDATION_COUNT:
        raise ValueError(
            f"Need more than {VALIDATION_COUNT} latent samples with rho; found {len(samples)}. "
            "Regenerate/export more samples with the updated rho-writing exporter."
        )
    return samples


def make_splits(samples, val_count: int = VALIDATION_COUNT, seed: int = RANDOM_SEED):
    samples = list(samples)
    rng = random.Random(seed)
    rng.shuffle(samples)
    return samples[val_count:], samples[:val_count]


def physics_stats(samples):
    vals = torch.tensor([[math.log(s.nu), math.log(s.rho)] for s in samples], dtype=torch.float32)
    mean = vals.mean(dim=0)
    std = vals.std(dim=0).clamp_min(1e-6)
    return mean, std


class NavierLatentDataset(torch.utils.data.Dataset):
    def __init__(self, samples, physics_mean, physics_std):
        self.samples = list(samples)
        self.physics_mean = physics_mean.float()
        self.physics_std = physics_std.float()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        latents = load_file(str(sample.latents_path))["latents"].float()
        physics = torch.tensor([math.log(sample.nu), math.log(sample.rho)], dtype=torch.float32)
        physics = (physics - self.physics_mean) / self.physics_std
        return {"latents": latents, "physics": physics, "sample_index": idx}


def collate_batch(items):
    return {
        "latents": torch.stack([x["latents"] for x in items]),
        "physics": torch.stack([x["physics"] for x in items]),
        "sample_index": torch.tensor([x["sample_index"] for x in items], dtype=torch.long),
    }


def latent_seq_len(latents: torch.Tensor, patch_size=(1, 2, 2)) -> int:
    _, _, f, h, w = latents.shape
    return (f // patch_size[0]) * (h // patch_size[1]) * (w // patch_size[2])


def null_context(batch_size: int, model, dtype=TRAIN_DTYPE):
    return [torch.zeros(1, model.text_dim, device=DEVICE, dtype=dtype) for _ in range(batch_size)]


def make_wan_scheduler(
    steps: int = VALIDATION_INFERENCE_STEPS,
    solver: str = WAN_SAMPLE_SOLVER,
    shift: float = WAN_SAMPLE_SHIFT,
    device: torch.device = DEVICE,
):
    if solver == "unipc":
        scheduler = FlowUniPCMultistepScheduler(
            num_train_timesteps=WAN_NUM_TRAIN_TIMESTEPS,
            shift=1,
            use_dynamic_shifting=False,
        )
        scheduler.set_timesteps(steps, device=device, shift=shift)
        return scheduler

    if solver in {"dpm++", "dpm"}:
        scheduler = FlowDPMSolverMultistepScheduler(
            num_train_timesteps=WAN_NUM_TRAIN_TIMESTEPS,
            shift=1,
            use_dynamic_shifting=False,
        )
        sampling_sigmas = get_sampling_sigmas(steps, shift)
        retrieve_timesteps(scheduler, device=device, sigmas=sampling_sigmas)
        return scheduler

    raise NotImplementedError(f"Unsupported Wan sample solver: {solver}")


def wan_inference_schedule(
    steps: int = VALIDATION_INFERENCE_STEPS,
    solver: str = WAN_SAMPLE_SOLVER,
    shift: float = WAN_SAMPLE_SHIFT,
    device: torch.device = DEVICE,
):
    scheduler = make_wan_scheduler(steps=steps, solver=solver, shift=shift, device=device)
    sigmas = scheduler.sigmas[:-1].to(device=device, dtype=torch.float32)
    timesteps = scheduler.timesteps.to(device=device)
    return sigmas, timesteps


def wan_inference_sigmas(
    steps: int = VALIDATION_INFERENCE_STEPS,
    solver: str = WAN_SAMPLE_SOLVER,
    shift: float = WAN_SAMPLE_SHIFT,
    device: torch.device = DEVICE,
):
    return wan_inference_schedule(steps=steps, solver=solver, shift=shift, device=device)[0]


def sample_wan_training_schedule(batch_size: int, device: torch.device = DEVICE):
    sigmas, timesteps = wan_inference_schedule(device=device)
    indices = torch.randint(0, sigmas.numel(), (batch_size,), device=device)
    return sigmas[indices].clamp(1e-5, 1.0 - 1e-5), timesteps[indices]


def apply_first_frame_condition(x: torch.Tensor, clean: torch.Tensor) -> torch.Tensor:
    x = x.clone()
    x[:, :, 0:1] = clean[:, :, 0:1]
    return x


def first_frame_condition_timesteps(latents: torch.Tensor, timesteps: torch.Tensor, patch_size) -> torch.Tensor:
    """Wan TI2V-style token timesteps: first latent frame is conditioning at t=0."""
    b, _, f, h, w = latents.shape
    pf, ph, pw = patch_size
    grid_f, grid_h, grid_w = f // pf, h // ph, w // pw
    token_timesteps = timesteps.to(device=latents.device).view(b, 1, 1, 1).expand(b, grid_f, grid_h, grid_w).clone()
    token_timesteps[:, 0] = 0
    return token_timesteps.flatten(1)


def flow_matching_batch(batch, model):
    clean = batch["latents"].to(device=DEVICE, dtype=TRAIN_DTYPE)
    physics = batch["physics"].to(device=DEVICE, dtype=torch.float32)
    b = clean.shape[0]
    noise = torch.randn_like(clean)
    sigmas, timesteps = sample_wan_training_schedule(b, device=DEVICE)
    view_shape = (b,) + (1,) * (clean.ndim - 1)
    x_t = (1.0 - sigmas.view(view_shape)) * clean + sigmas.view(view_shape) * noise
    x_t = apply_first_frame_condition(x_t, clean)
    target = noise - clean
    target[:, :, 0:1] = 0.0

    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    seq_len = latent_seq_len(clean, base.patch_size)
    timestep_tokens = first_frame_condition_timesteps(clean, timesteps, base.patch_size)
    contexts = null_context(b, base)

    with torch.amp.autocast("cuda", dtype=TRAIN_DTYPE):
        pred = model(
            [x_t[i] for i in range(b)],
            timestep_tokens,
            context=contexts,
            seq_len=seq_len,
            physics=physics,
        )
        pred = torch.stack(pred, dim=0)
        loss = F.mse_loss(pred[:, :, 1:], target[:, :, 1:].float())
    return loss


def trainable_parameters(model):
    return [p for p in model.parameters() if p.requires_grad]


def learning_rate_for_step(step: int) -> float:
    if LR_WARMUP_STEPS > 0 and step <= LR_WARMUP_STEPS:
        warmup_fraction = max(0.0, float(step) / float(LR_WARMUP_STEPS))
        return float(LEARNING_RATE) * warmup_fraction
    if MAX_STEPS <= LR_WARMUP_STEPS:
        return float(LEARNING_RATE)
    decay_progress = (float(step) - float(LR_WARMUP_STEPS)) / float(MAX_STEPS - LR_WARMUP_STEPS)
    decay_progress = min(1.0, max(0.0, decay_progress))
    cosine_scale = 0.5 * (1.0 + math.cos(math.pi * decay_progress))
    return float(LEARNING_RATE) * cosine_scale


def set_optimizer_lr(optimizer, lr: float) -> None:
    for group in optimizer.param_groups:
        group["lr"] = float(lr)


def build_training_objects():
    samples = discover_samples()
    train_samples, val_samples = make_splits(samples)
    p_mean, p_std = physics_stats(train_samples)
    train_ds = NavierLatentDataset(train_samples, p_mean, p_std)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        collate_fn=collate_batch,
        drop_last=True,
    )

    transformer = load_wan_transformer()
    transformer = attach_physics_adaln_forward(transformer)
    model = add_lora_to_wan(transformer)
    model.train()
    optimizer = torch.optim.AdamW(trainable_parameters(model), lr=LEARNING_RATE, weight_decay=1e-4)

    try:
        model.print_trainable_parameters()
    except Exception:
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"trainable params: {trainable:,} / {total:,}")

    return model, optimizer, train_loader, train_samples, val_samples, p_mean, p_std


def load_wan_vae(dtype=torch.float32, device=DEVICE):
    Wan2_2_VAE = load_wan_vae_class()
    return Wan2_2_VAE(
        vae_pth=str(WAN_CHECKPOINT_DIR / "Wan2.2_VAE.pth"),
        dtype=dtype,
        device=device,
    )


@torch.no_grad()
def infer_latents_from_first_frame(model, clean_latents, physics, steps=VALIDATION_INFERENCE_STEPS):
    model.eval()
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    clean = clean_latents.to(device=DEVICE, dtype=TRAIN_DTYPE)
    z = torch.randn_like(clean)
    z[:, 0:1] = clean[:, 0:1]
    physics = physics.to(device=DEVICE, dtype=torch.float32).unsqueeze(0)
    seq_len = latent_seq_len(clean.unsqueeze(0), base.patch_size)
    context = null_context(1, base)

    scheduler = make_wan_scheduler(steps=steps, device=DEVICE)
    for t in scheduler.timesteps:
        timestep = torch.stack([t]).to(device=DEVICE)
        timestep_tokens = first_frame_condition_timesteps(clean.unsqueeze(0), timestep, base.patch_size)
        with torch.amp.autocast("cuda", dtype=TRAIN_DTYPE):
            pred = model([z], timestep_tokens, context=context, seq_len=seq_len, physics=physics)[0]
        z = scheduler.step(
            pred.unsqueeze(0),
            t,
            z.unsqueeze(0),
            return_dict=False,
        )[0].squeeze(0).to(dtype=TRAIN_DTYPE)
        z[:, 0:1] = clean[:, 0:1]

    model.train()
    return z.float().cpu()


@torch.no_grad()
def decode_latents_to_video(vae, latents):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")
        video = vae.decode([latents.to(device=DEVICE, dtype=torch.float32)])[0]
    return video.detach().float().cpu().clamp(-1, 1)


def video_cthw_to_uint8_frames(video: torch.Tensor, output_size: int = 256) -> torch.Tensor:
    frames = ((video.float().clamp(-1, 1) + 1.0) * 0.5).permute(1, 0, 2, 3).contiguous()
    if frames.shape[-2:] != (int(output_size), int(output_size)):
        frames = F.interpolate(frames, size=(int(output_size), int(output_size)), mode="bilinear", align_corners=False)
    return (255.0 * frames.permute(0, 2, 3, 1)).round().clamp(0, 255).to(torch.uint8).cpu()


def error_to_uint8_frames(error: torch.Tensor, vmax: float = VALIDATION_ERROR_VMAX) -> torch.Tensor:
    x = (error.float() / max(float(vmax), 1e-12)).clamp(0.0, 1.0)
    red = x
    green = (1.35 * x - 0.35).clamp(0.0, 1.0)
    blue = (2.0 * x - 1.5).clamp(0.0, 1.0)
    return (255.0 * torch.stack((red, green, blue), dim=-1)).round().clamp(0, 255).to(torch.uint8).cpu()


def validation_triplet_frames(gt_video, pred_video, output_size: int = 256) -> torch.Tensor:
    gt = video_cthw_to_uint8_frames(gt_video, output_size=output_size)
    pred = video_cthw_to_uint8_frames(pred_video, output_size=output_size)
    frame_count = min(gt.shape[0], pred.shape[0])
    gt = gt[:frame_count]
    pred = pred[:frame_count]
    error = (gt.float() / 255.0 - pred.float() / 255.0).abs().mean(dim=-1)
    err = error_to_uint8_frames(error)
    return torch.cat((gt, pred, err), dim=2).contiguous()


def encode_rgb_frames_mp4(frames: torch.Tensor, fps: int = 12) -> bytes:
    if frames.ndim != 4 or frames.shape[-1] != 3:
        raise ValueError("frames must have shape (time, height, width, 3)")
    frames = frames.detach().to(torch.uint8).cpu()
    buffer = io.BytesIO()
    with av.open(buffer, mode="w", format="mp4") as container:
        stream = container.add_stream("libx264", rate=int(fps))
        stream.width = int(frames.shape[2])
        stream.height = int(frames.shape[1])
        stream.pix_fmt = "yuv420p"
        stream.options = {"crf": "18", "preset": "veryfast"}
        for frame in frames.numpy():
            video_frame = av.VideoFrame.from_ndarray(frame, format="rgb24")
            for packet in stream.encode(video_frame):
                container.mux(packet)
        for packet in stream.encode():
            container.mux(packet)
    return buffer.getvalue()


def display_validation_triplet(gt_video, pred_video, title="validation", output_size: int = 256, fps: int = 12):
    del title
    frames = validation_triplet_frames(gt_video, pred_video, output_size=output_size)
    video_b64 = base64.b64encode(encode_rgb_frames_mp4(frames, fps=fps)).decode("ascii")
    height = int(frames.shape[1])
    width = int(frames.shape[2])
    html = HTML(
        f'<video autoplay loop muted playsinline controls '
        f'width="{width}" height="{height}" '
        f'style="display:block;width:{width}px;height:{height}px;margin:0;padding:0;border:0;line-height:0">'
        f'<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">'
        f'</video>'
    )
    display(html)
    return html


@torch.no_grad()
def validate_random_sample(model, vae, val_samples, physics_mean, physics_std, step):
    sample = random.choice(val_samples)
    clean_latents = load_file(str(sample.latents_path))["latents"].float()
    raw_physics = torch.tensor([math.log(sample.nu), math.log(sample.rho)], dtype=torch.float32)
    physics = (raw_physics - physics_mean.float()) / physics_std.float()
    pred_latents = infer_latents_from_first_frame(model, clean_latents, physics)
    gt_video = decode_latents_to_video(vae, clean_latents)
    pred_video = decode_latents_to_video(vae, pred_latents)
    title = f"step {step} | {sample.folder.name} | nu={sample.nu:.2e}, rho={sample.rho:.3g}"
    return display_validation_triplet(gt_video, pred_video, title=title)


def train():
    model, optimizer, train_loader, train_samples, val_samples, physics_mean, physics_std = build_training_objects()
    vae = load_wan_vae(dtype=torch.float32, device=DEVICE)
    loader_iter = iter(train_loader)
    running = []
    ema_loss = None

    optimizer.zero_grad(set_to_none=True)
    progress = trange(1, MAX_STEPS + 1, desc="fine-tuning", dynamic_ncols=True)
    for step in progress:
        lr = learning_rate_for_step(step)
        set_optimizer_lr(optimizer, lr)
        accum_loss = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch = next(loader_iter)

            loss = flow_matching_batch(batch, model) / GRAD_ACCUM_STEPS
            loss.backward()
            accum_loss += float(loss.detach().cpu())

        torch.nn.utils.clip_grad_norm_(trainable_parameters(model), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        running.append(accum_loss)
        ema_loss = accum_loss if ema_loss is None else LOSS_EMA_BETA * ema_loss + (1.0 - LOSS_EMA_BETA) * accum_loss

        progress.set_postfix(
            loss=f"{accum_loss:.6f}",
            ema=f"{ema_loss:.6f}",
            avg10=f"{sum(running[-10:]) / min(10, len(running)):.6f}",
            lr=f"{lr:.2e}",
        )

        if step % VALIDATION_EVERY == 0:
            validate_random_sample(model, vae, val_samples, physics_mean, physics_std, step)

    return model


# Run this cell, then start fine-tuning with:
model = train()


/opt/miniforge/envs/new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  2.70it/s]


trainable params: 26,606,400 || all params: 5,026,394,112 || trainable%: 0.5293


fine-tuning:   2%|▏         | 999/40000 [05:08<3:07:44,  3.46it/s, avg10=0.086091, ema=0.104974, loss=0.087847, lr=3.00e-05]

fine-tuning:   5%|▍         | 1999/40000 [10:27<3:01:41,  3.49it/s, avg10=0.100544, ema=0.088822, loss=0.077971, lr=2.99e-05] 

fine-tuning:   7%|▋         | 2903/40000 [15:17<3:15:25,  3.16it/s, avg10=0.104010, ema=0.094544, loss=0.070929, lr=2.97e-05] 


KeyboardInterrupt: 